In [1]:
!pip install pymongo pandas matplotlib requests beautifulsoup4 kafka-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 31.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
from pymongo import MongoClient

# 1. Leer el CSV
df_kaggle = pd.read_csv('steam_kaggle.csv', low_memory=False)

# LIMPIEZA DE COLUMNAS: Quitamos espacios en blanco al principio o final de los nombres
df_kaggle.columns = df_kaggle.columns.str.strip()

print("Columnas detectadas:", df_kaggle.columns.tolist())

# 2. Selección de columnas (asegúrate de que coincidan con tu lista)
cols = ['Name', 'Release date', 'Price', 'Positive', 'Negative', 'Genres', 'Average playtime forever']
# Solo seleccionamos las columnas que realmente existan en el CSV para evitar errores
cols_existentes = [c for c in cols if c in df_kaggle.columns]
df_limpio = df_kaggle[cols_existentes].copy()

# 3. Tratamiento de Fechas (más robusto)
# Intentamos convertir, y lo que no se entienda lo dejamos como "01-01-2000" para no perder la fila
df_limpio['Release date'] = pd.to_datetime(df_limpio['Release date'], errors='coerce')
df_limpio['Release date'] = df_limpio['Release date'].fillna(pd.Timestamp('2000-01-01'))

# 4. Verificación antes de insertar
if df_limpio.empty:
    print("ERROR: El DataFrame está vacío. Revisa el nombre de las columnas.")
else:
    # Convertimos a diccionario para Mongo
    documentos = df_limpio.to_dict('records')
    
    if len(documentos) > 0:
        client = MongoClient('mongodb://mongodb:27017')
        db = client['steam_db']
        
        db.juegos_historicos.drop() 
        db.juegos_historicos.insert_many(documentos)
        print(f"{len(documentos)} juegos de Kaggle cargados en MongoDB.")
    else:
        print("ERROR: No hay documentos para insertar.")

Columnas detectadas: ['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'DiscountDLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']


/tmp/ipykernel_5101/1355501032.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_limpio['Release date'] = pd.to_datetime(df_limpio['Release date'], errors='coerce')


122611 juegos de Kaggle cargados en MongoDB.
